# Break → Fix

*0.1 Python for GenAI · run **Setup** first*

## Setup

Settings, a configured client, and two helpers. Every cell below uses them.

In [1]:
"""Shared setup for this notebook: typed settings, configured clients, logging."""

import asyncio
import json
import logging
from concurrent.futures import ThreadPoolExecutor

from dotenv import find_dotenv
from openai import AsyncOpenAI, OpenAI
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    """All configuration in one validated object, read from the environment / .env."""

    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()

client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value(),
    timeout=settings.request_timeout_seconds,
    max_retries=settings.max_retries,
)


def async_client() -> AsyncOpenAI:
    """A fresh async client per event loop (async clients are bound to the loop they run in)."""
    return AsyncOpenAI(
        api_key=settings.openai_api_key.get_secret_value(),
        timeout=settings.request_timeout_seconds,
        max_retries=settings.max_retries,
    )


def run_async(coroutine):
    """Run a coroutine from a notebook (which already has an event loop). Scripts use asyncio.run()."""
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coroutine).result()


def show(title: str, value) -> None:
    """Print a labelled, formatted JSON block."""
    print(title)
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
for noisy in ["httpx", "httpx2", "httpcore", "openai"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)

print("model:", settings.openai_model, "| timeout:", settings.request_timeout_seconds, "s")

model: gpt-4o-mini | timeout: 30.0 s


### Break → Fix

> **Problem.** A batch job has `await` inside a `for` loop. It looks concurrent — it uses the async client, it has `await` everywhere — and it runs one request at a time. It works, so nobody notices, until the batch grows from 100 items to 10,000 and the job takes three hours.

**Idea.** Build the requests first, run them with `gather`, cap how many run at once.

**Use when** you see run time growing in a straight line with item count while CPU sits idle.  
**Not when** —.

```
for p in prompts:                          await ──▶ await ──▶ await ──▶ …    12 × 0.4 s
    answers.append(await ask(p))

tasks = [ask(p) for p in prompts]          ┌ await ┐
await gather(*tasks)                       ├ await ┤  together                 ≈ 0.6 s
                                           └ await ┘
```

**How it works.**
1. In the broken version each `await ask(prompt)` finishes completely before the loop moves on — the loop *is* the serialisation.
2. The fix builds every coroutine first without awaiting, then hands the list to `gather`, which runs them together.
3. A `Semaphore(6)` inside the fixed version caps in-flight requests, so the fix does not turn a slow job into a rate-limited one.
4. Both versions are timed on the same 12 prompts and compared; the answers are identical.

| | what happens | result |
|:--|:--|:--|
| ✗ await in loop | 12 requests one at a time | ≈12 × one call |
| ✓ gather + semaphore | 12 requests, 6 at once | ≈2 × one call |

**Production code and its real output**

In [2]:
# Break → Fix. BREAK: a batch job that awaits each call inside a loop (no concurrency).
# FIX: gather under a semaphore. Same answers, a fraction of the wall-clock time.
import time

PROMPTS = []
for n in range(1, 13):
    PROMPTS.append(f"What is {n} times 3? Number only.")


async def ask(ai: AsyncOpenAI, prompt: str) -> str:
    response = await ai.chat.completions.create(
        model=settings.openai_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=4,
    )
    return response.choices[0].message.content.strip()


async def broken() -> list[str]:
    async with async_client() as ai:
        answers = []
        for prompt in PROMPTS:
            answers.append(await ask(ai, prompt))
        return answers


async def fixed() -> list[str]:
    semaphore = asyncio.Semaphore(6)
    async with async_client() as ai:

        async def limited(prompt: str) -> str:
            async with semaphore:
                return await ask(ai, prompt)

        tasks = []
        for prompt in PROMPTS:
            tasks.append(limited(prompt))
        return await asyncio.gather(*tasks)


started = time.perf_counter()
broken_answers = run_async(broken())
broken_seconds = time.perf_counter() - started
started = time.perf_counter()
fixed_answers = run_async(fixed())
fixed_seconds = time.perf_counter() - started
print(f"BREAK await in a loop:    {broken_seconds:.2f}s")
print(f"FIX   gather + semaphore: {fixed_seconds:.2f}s")
assert broken_answers == fixed_answers and fixed_seconds < broken_seconds

BREAK await in a loop:    6.53s
FIX   gather + semaphore: 1.73s


**What the output shows.** Same answers, several times faster. The ratio grows with the batch size — at 10,000 items it is the difference between hours and minutes.

**In practice**
- **the fingerprint** — run time proportional to item count with idle CPU is always this bug or a missing index; check the loop first.
- **cap when fixing** — adding `gather` without a limit moves the failure from slow to 429; add the semaphore in the same change.
- **code review** — `await` directly inside a `for` over independent items is worth a comment every time; it is almost never intended.

**Alternatives** — `asyncio.TaskGroup` · a job queue with workers

**Terms** — *linear scaling*: time grows in proportion to items — the fingerprint of no concurrency
